
# Melbourne housing price  prediction - Comprehensive Model Comparison






## ABOUT:


About Dataset
Context
Melbourne real estate is BOOMING. Can you find the insight or predict the next big trend to become a real estate mogul… or even harder, to snap up a reasonably priced 2-bedroom unit?

Content
This is a snapshot of a dataset created by Tony Pino.

It was scraped from publicly available results posted every week from Domain.com.au. He cleaned it well, and now it's up to you to make data analysis magic. The dataset includes Address, Type of Real estate, Suburb, Method of Selling, Rooms, Price, Real Estate Agent, Date of Sale and distance from C.B.D.

Notes on Specific Variables
Rooms: Number of rooms

Price: Price in dollars

Method: S - property sold; SP - property sold prior; PI - property passed in; PN - sold prior not disclosed; SN - sold not disclosed; NB - no bid; VB - vendor bid; W - withdrawn prior to auction; SA - sold after auction; SS - sold after auction price not disclosed. N/A - price or highest bid not available.

Type: br - bedroom(s); h - house,cottage,villa, semi,terrace; u - unit, duplex; t - townhouse; dev site - development site; o res - other residential.

SellerG: Real Estate Agent

Date: Date sold

Distance: Distance from CBD

Regionname: General Region (West, North West, North, North east …etc)

Propertycount: Number of properties that exist in the suburb.

Bedroom2 : Scraped # of Bedrooms (from different source)

Bathroom: Number of Bathrooms

Car: Number of carspots

Landsize: Land Size

BuildingArea: Building Size

CouncilArea: Governing council for the area

Acknowledgements
This is intended as a static (unchanging) snapshot of https://www.kaggle.com/anthonypino/melbourne-housing-market. It was created in September 2017. Additionally, homes with no Price have been removed.

## Import Libraries
This cell imports all the required Python libraries for data handling, modeling, and evaluation.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc,mean_absolute_error, mean_squared_error, r2_score
import joblib

plt.style.use("default")
plt.rcParams["figure.figsize"] = (5, 3)


## Loading data

Here we load the cleaned Pima Indians Diabetes dataset using pd.read.



In [ ]:
data = pd.read_csv("melb_data.csv", on_bad_lines="skip")
data.head()





In [ ]:
data.describe()


## Data Cleaning

We remove rows with missing target values ("Price") and drop columns that add noise or arent not useful for modelling.  
We also filter extreme outliers in land size to improve model performance.


In [ ]:
data = data.dropna()
data = data[data["Landsize"] < data["Landsize"].quantile(0.99)]
data = data.dropna(subset=["Price"])
data = data.drop(columns=["Address", "SellerG", "Method", "Date", "Latitude", "Longitude", "CouncilArea","Postcode", "CouncilArea", "Bedroom2", "Propertycount"], errors="ignore")



data.info()


In [ ]:

from sklearn.model_selection import train_test_split

X = data.drop(['Price'], axis = 1)
y = data['Price']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
train_data = X_train.join(y_train)


In [ ]:
train_data.hist(bins=50, figsize=(20,12))

In [ ]:
sns.histplot(data["Price"], kde=True)
plt.title("Distribution of House Prices")
plt.show()

In [ ]:
sns.scatterplot(x=data["Rooms"], y=data["Price"])
plt.title("Rooms vs Price")
plt.show()

sns.scatterplot(x=data["Landsize"], y=data["Price"])
plt.title("Landsize vs Price")
plt.show()

## Feature selection and pre-processing

After cleaning the dataset, we identify the features most relevant for modelling.  
Categorical features (such as "Suburb") require one-hot encoding to turn them into a binary matrix, numerical data is kept the same.

Using a scikit-learn "ColumnTransformer" we built a unified preprocessing pipeline where numeric features are imputed using the median and scaled.
Categorical features are imputed with the most frequent value and then one-hot encoded.




In [ ]:
numeric_data = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_data = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric:", numeric_data)
print("Categorical:", categorical_data)
print(type(numeric_data), type(categorical_data))


In [ ]:
plt.figure(figsize=(12,10))
sns.scatterplot(x='BuildingArea', y='Distance', hue='Price', data=train_data, palette='coolwarm', alpha=0.6)

In [ ]:

plt.figure(figsize=(14, 8))
sns.heatmap(data[numeric_data].corr(), annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()


In [ ]:

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_data),
        ("cat", categorical_transformer, categorical_data),
        ])

In [ ]:
type(X_train)


In [ ]:
model_lin = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

model_lin.fit(X_train, y_train)



Model and Model Comparison

We train two regression algorithms are trained to compare predictive performance:

1. **Linear Regression** 
2. **Random Forest Regressor** 

Both models share the same preprocessing pipeline for a fair comparison.  
After training both are evaluated on the test set using identical performance metrics.

In [ ]:
y_pred_lin = model_lin.predict(X_test)

mae_lin = mean_absolute_error(y_test, y_pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred_lin))
r2_lin = r2_score(y_test, y_pred_lin)


print("Linear Regression:")
print("MAE:", mae_lin)
print("RMSE:", rmse_lin)
print("R2:", r2_lin)

# Random Forest Regressor


In [ ]:
model_rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ))
])

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest:")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)

Evaluation

We assessed model performance using three widely accepted regression metrics:

- **MAE (Mean Absolute Error):** 
- **RMSE (Root Mean Squared Error):** 
- **R² (Coefficient of Determination):**

These metrics allow comparsion between the baseline Linear Regression model and the more advanced Random Forest model.  
A lower MAE and RMSE, and a higher R², indicate better performance.


In [ ]:

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lin, mae_rf],
    "RMSE": [rmse_lin, rmse_rf],
    "R2": [r2_lin, r2_rf]
})

results


Saving the Final Model

We collect all validation accuracies, identify the best-performing model, and save it to disk using "joblib" for later use.

In [ ]:
import joblib


best_model = model_rf 


best_model.fit(X, y)    

joblib.dump(best_model, "melbourne_house_model.pkl")
print("Model saved as melbourne_house_model.pkl")


In [ ]:
import joblib
import pandas as pd

model = joblib.load("melbourne_house_model.pkl")

example_house = X_test.iloc[[0]].copy()

print("Example house features:")
display(example_house)

predicted_price = model.predict(example_house)[0]
print(f"Predicted Price: ${predicted_price:.0f}")
actual_price = y_test.iloc[0]
print(f"Actual Price: ${actual_price:.0f}")

